In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.2f}'.format)

DATA_PATH = os.path.expanduser("~/Desktop/europrocure-intelligence/data/raw/export_CAN_2023_2018.csv")

# Read the full dataset in chunks and sample evenly across years
# This ensures all 6 years are represented fairly
print("Reading full dataset in chunks — this will take 2-3 minutes...")

chunks = []
chunk_size = 100000
total_rows = 0

for chunk in pd.read_csv(DATA_PATH, chunksize=chunk_size, encoding='utf-8', low_memory=False):
    chunks.append(chunk)
    total_rows += len(chunk)
    print(f"  Read {total_rows:,} rows so far...", end='\r')

df_full = pd.concat(chunks, ignore_index=True)
print(f"\nFull dataset loaded: {len(df_full):,} rows and {len(df_full.columns)} columns")

# Check year distribution in full dataset
print("\nYear distribution in full dataset:")
print(df_full['YEAR'].value_counts().sort_index().to_string())

Reading full dataset in chunks — this will take 2-3 minutes...
  Read 6,198,063 rows so far...
Full dataset loaded: 6,198,063 rows and 75 columns

Year distribution in full dataset:
YEAR
2018     804040
2019     990811
2020    1070272
2021    1162663
2022    1071826
2023    1098451


In [4]:
# Deduplicated contract counts by year — the real picture
year_counts = df_full.groupby('YEAR').agg(
    unique_contracts=('ID_NOTICE_CAN', 'nunique'),
    total_rows=('ID_NOTICE_CAN', 'count')
).reset_index()

year_counts['avg_rows_per_contract'] = (
    year_counts['total_rows'] / year_counts['unique_contracts']
).round(2)

print("Full dataset — contracts by year:")
print(year_counts.to_string(index=False))

fig = px.bar(
    year_counts,
    x='YEAR',
    y='unique_contracts',
    title='Unique Contracts by Year — Full TED Dataset 2018-2023',
    labels={'unique_contracts': 'Number of Unique Contracts', 'YEAR': 'Year'},
    color='unique_contracts',
    color_continuous_scale='Blues',
    text='unique_contracts'
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    xaxis=dict(tickmode='linear')
)
fig.update_traces(textposition='outside')
fig.show()

Full dataset — contracts by year:
 YEAR  unique_contracts  total_rows  avg_rows_per_contract
 2018            236266      804040                   3.40
 2019            264771      990811                   3.74
 2020            267596     1070272                   4.00
 2021            285506     1162663                   4.07
 2022            300617     1071826                   3.57
 2023            305470     1098451                   3.60


In [5]:
# Procedure type distribution by year — core of our COVID hypothesis
procedure_by_year = df_full.groupby(['YEAR', 'TOP_TYPE']).agg(
    contracts=('ID_NOTICE_CAN', 'nunique')
).reset_index()

# Calculate percentage within each year
procedure_by_year['total_year'] = procedure_by_year.groupby('YEAR')['contracts'].transform('sum')
procedure_by_year['pct'] = (procedure_by_year['contracts'] / procedure_by_year['total_year'] * 100).round(2)

print("Procedure type by year (% of contracts):")
pivot = procedure_by_year.pivot(index='YEAR', columns='TOP_TYPE', values='pct').fillna(0)
print(pivot.round(2).to_string())

fig = px.bar(
    procedure_by_year,
    x='YEAR',
    y='pct',
    color='TOP_TYPE',
    title='Procedure Type Distribution by Year — Did COVID change procurement methods?',
    labels={'pct': '% of Contracts', 'YEAR': 'Year', 'TOP_TYPE': 'Procedure'},
    barmode='stack',
    text_auto='.1f'
)

fig.update_layout(
    plot_bgcolor='white',
    xaxis=dict(tickmode='linear')
)
fig.show()

Procedure type by year (% of contracts):
TOP_TYPE  AWP  COD  INP  NIC  NIP  NOC  NOP   OPE  RES
YEAR                                                  
2018     1.88 0.26 0.02 8.96 0.20 4.58 0.26 79.48 4.35
2019     1.86 0.23 0.03 8.29 0.19 4.46 0.26 80.07 4.61
2020     2.15 0.21 0.03 7.85 0.22 6.55 0.28 78.34 4.37
2021     2.12 0.21 0.02 7.43 0.22 5.66 0.24 79.25 4.84
2022     2.51 0.19 0.02 7.24 0.21 6.21 0.24 78.20 5.19
2023     3.18 0.18 0.01 6.67 0.19 5.65 0.20 78.24 5.67


In [6]:
# Contract value distribution — VALUE_EURO_FIN_2
# Remove nulls and obvious outliers for visualisation
value_data = df_full['VALUE_EURO_FIN_2'].dropna()
value_data = value_data[value_data > 0]

# Cap at 99th percentile for visualisation — outliers distort the chart
p99 = value_data.quantile(0.99)
value_capped = value_data[value_data <= p99]

print(f"VALUE_EURO_FIN_2 statistics:")
print(f"  Total non-null rows : {len(value_data):,}")
print(f"  Coverage            : {len(value_data)/len(df_full)*100:.1f}%")
print(f"  Min                 : €{value_data.min():,.0f}")
print(f"  Median              : €{value_data.median():,.0f}")
print(f"  Mean                : €{value_data.mean():,.0f}")
print(f"  99th percentile     : €{p99:,.0f}")
print(f"  Max                 : €{value_data.max():,.0f}")

# Value by year
value_by_year = df_full[df_full['VALUE_EURO_FIN_2'] > 0].groupby('YEAR').agg(
    median_value=('VALUE_EURO_FIN_2', 'median'),
    mean_value=('VALUE_EURO_FIN_2', 'mean'),
    total_value=('VALUE_EURO_FIN_2', 'sum')
).reset_index()

value_by_year['total_value_bn'] = (value_by_year['total_value'] / 1e9).round(2)

print(f"\nContract value by year:")
print(value_by_year[['YEAR', 'median_value', 'mean_value', 'total_value_bn']].to_string(index=False))

VALUE_EURO_FIN_2 statistics:
  Total non-null rows : 5,630,386
  Coverage            : 90.8%
  Min                 : €0
  Median              : €343,745
  Mean                : €13,158,471
  99th percentile     : €103,891,989
  Max                 : €10,000,000,000,000

Contract value by year:
 YEAR  median_value  mean_value  total_value_bn
 2018     376569.65  4888118.16         3534.04
 2019     359562.54  6909065.69         6230.55
 2020     306779.83  5722950.14         5607.20
 2021     286149.00 10526650.81        11232.01
 2022     380001.46  7962360.74         7680.10
 2023     386216.20 40032522.75        39803.38


In [7]:
# Top 15 countries by number of unique contracts
country_counts = df_full.groupby('ISO_COUNTRY_CODE').agg(
    unique_contracts=('ID_NOTICE_CAN', 'nunique'),
    total_value_bn=('VALUE_EURO_FIN_2', lambda x: x.sum() / 1e9)
).reset_index().sort_values('unique_contracts', ascending=False).head(15)

print("Top 15 countries by contract volume:")
print(country_counts.to_string(index=False))

fig = px.bar(
    country_counts,
    x='ISO_COUNTRY_CODE',
    y='unique_contracts',
    title='Top 15 Countries by Number of Contracts — 2018-2023',
    labels={'unique_contracts': 'Unique Contracts', 'ISO_COUNTRY_CODE': 'Country'},
    color='unique_contracts',
    color_continuous_scale='Blues',
    text='unique_contracts'
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False
)
fig.update_traces(textposition='outside')
fig.show()

Top 15 countries by contract volume:
ISO_COUNTRY_CODE  unique_contracts  total_value_bn
              DE            286538        11222.08
              FR            224471         7636.35
              PL            192256         1882.48
              ES            122301         3160.48
              RO             99588         2295.33
              CZ             91219          351.53
              UK             70370        12657.99
              BG             67474          185.53
              IT             65256         5210.38
              SE             51266         1119.96
              NL             43731        10425.06
              SI             33681          270.02
              LT             28259          267.26
              BE             25976          407.48
              FI             25543          293.38


In [8]:
# Top CPV divisions by contract volume
# Extract first 2 digits of CPV code = procurement division
df_full['cpv_division'] = df_full['CPV'].astype(str).str[:2]

cpv_counts = df_full.groupby('cpv_division').agg(
    unique_contracts=('ID_NOTICE_CAN', 'nunique')
).reset_index().sort_values('unique_contracts', ascending=False).head(15)

# CPV division labels — top categories
cpv_labels = {
    '45': 'Construction works',
    '72': 'IT services',
    '79': 'Business services',
    '71': 'Architectural services',
    '33': 'Medical equipment',
    '34': 'Transport equipment',
    '90': 'Sewage & waste services',
    '50': 'Repair & maintenance',
    '48': 'Software packages',
    '60': 'Transport services',
    '64': 'Postal services',
    '85': 'Health services',
    '44': 'Construction structures',
    '55': 'Hotel & restaurant services',
    '32': 'Radio & TV equipment'
}

cpv_counts['category_name'] = cpv_counts['cpv_division'].map(cpv_labels).fillna('Other')
cpv_counts['label'] = cpv_counts['cpv_division'] + ' — ' + cpv_counts['category_name']

print("Top 15 CPV divisions by contract volume:")
print(cpv_counts[['cpv_division', 'category_name', 'unique_contracts']].to_string(index=False))

fig = px.bar(
    cpv_counts,
    x='unique_contracts',
    y='label',
    orientation='h',
    title='Top 15 Procurement Categories (CPV Divisions) — 2018-2023',
    labels={'unique_contracts': 'Unique Contracts', 'label': 'CPV Category'},
    color='unique_contracts',
    color_continuous_scale='Blues',
    text='unique_contracts'
)

fig.update_layout(
    plot_bgcolor='white',
    coloraxis_showscale=False,
    yaxis={'categoryorder': 'total ascending'},
    height=600
)
fig.update_traces(textposition='outside')
fig.show()

Top 15 CPV divisions by contract volume:
cpv_division           category_name  unique_contracts
          45      Construction works            227887
          33       Medical equipment            190700
          71  Architectural services            142306
          90 Sewage & waste services            103261
          79       Business services             90446
          34     Transport equipment             77773
          50    Repair & maintenance             74582
          72             IT services             68181
          85         Health services             57188
          30                   Other             51613
          60      Transport services             44630
          77                   Other             38384
          38                   Other             37286
          39                   Other             35177
          15                   Other             34267


## EDA Distributions — Key Findings

### Contract Volume
- 1.66 million unique contracts across 2018-2023
- Volume grew steadily — COVID did NOT reduce contract numbers
- Average 3.4-4.1 rows per notice (lot-level duplication confirmed)

### COVID Impact — Revised Hypothesis
- Open procedure share stayed stable at ~78-80% throughout all years
- COVID reduced CONTRACT SIZE not contract volume
- Median value dropped from €376k (2018) to €286k (2021) — a 24% reduction
- Direct awards (AWP) grew 69% from 2018 to 2023 — a slow structural shift

### Contract Values
- VALUE_EURO_FIN_2 coverage: 90.8% — excellent
- Max value is €10 trillion — confirmed data entry error, must cap in cleaning
- Netherlands total value distorted by this single outlier

### Top Countries
- Germany (286k), France (224k), Poland (192k) — top 3 by volume

### Top Categories
- Construction (45), Medical equipment (33), Architectural services (71)
- IT services (72) — highest vendor concentration risk despite lower volume

### Actions for Day 3 Cleaning
1. Cap VALUE_EURO_FIN_2 at 99th percentile (€103M) to remove outliers
2. Extract cpv_division as first 2 digits of CPV
3. Clean WIN_COUNTRY_CODE — split on --- and take first value
4. Parse DT_DISPATCH and DT_AWARD as DD/MM/YY dates
5. Drop 8 columns confirmed above 70% missing
6. Deduplicate on ID_NOTICE_CAN for notice-level analysis